# Droppy Somalia: Gemma 4 Vision Fine-Tuning Prototype for Orange Jugs and Queues

This notebook provides a complete prototype for:
1.  **GPU Pre-flight Check** (Ensuring a T4 GPU is active).
2.  **Scraping images** (using the updated `ddgs` package).
3.  **Preparing a multimodal dataset** (with manual upload fallback).
4.  **Fine-tuning Gemma 4 E2B Vision** using QLoRA and the Unsloth library on a single T4 GPU.

## 0. GPU Pre-flight Check
**IMPORTANT**: Unsloth requires a GPU to run. If the cell below says "No GPU detected", please go to `Runtime > Change runtime type` and select `T4 GPU`.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[✓] GPU Detected: {torch.cuda.get_device_name(0)}")
    print("    You are ready to proceed!")
else:
    print("[✗] No GPU detected!")
    print("    Please go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.")

## 1. Install Dependencies

In [ ]:
# Install Unsloth and other necessary libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.13.0" peft accelerate bitsandbytes
!pip install ddgs Pillow requests

## 2. Resilient Image Scraping Utility

This utility uses the updated `ddgs` package. If you still encounter rate limits (403 error), proceed to Module 3 to upload images manually.

In [ ]:
import os
import requests
import time
import random
from ddgs import DDGS
from PIL import Image
from io import BytesIO

def download_images(keywords, max_images=5, output_dir="dataset"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print(f"Searching for: {keywords}...")
    image_paths = []
    try:
        with DDGS() as ddgs:
            results = list(ddgs.images(keywords, max_results=max_images))
            count = 0
            for i, res in enumerate(results):
                try:
                    time.sleep(random.uniform(1.0, 3.0))
                    response = requests.get(res['image'], timeout=10)
                    img = Image.open(BytesIO(response.content)).convert("RGB")
                    path = os.path.join(output_dir, f"{keywords.replace(' ', '_')}_{count}.jpg")
                    img.save(path)
                    image_paths.append(path)
                    count += 1
                    print(f"  [✓] Downloaded: {path}")
                except Exception as e:
                    print(f"  [✗] Failed to download {res['image']}: {e}")
    except Exception as e:
        print(f"[!] Scraper Error (likely rate-limited): {e}")
        print("    Proceed to Module 3 for manual upload instructions.")
    return image_paths

# Attempt to scrape prototype images
orange_jug_images = download_images("orange plastic water jug", max_images=3)
queue_images = download_images("water queue Africa", max_images=3)

## 3. Manual Image Upload Fallback

If the scraper failed due to rate limits, follow these steps:
1.  Click the **Folder icon** 📁 on the left sidebar of Colab.
2.  Right-click and select **New folder**, name it `dataset`.
3.  Right-click the `dataset` folder and select **Upload**.
4.  Upload at least 2-3 `.jpg` images of orange jugs and water queues.
5.  Run the cell below.

In [ ]:
import glob
scraped_images = glob.glob("dataset/*.jpg")
if not scraped_images:
    print("No images found in 'dataset/'. Please follow the manual upload instructions above.")
else:
    print(f"Found {len(scraped_images)} images for training.")
dataset = []
for path in scraped_images:
    if "orange" in path.lower() or "jug" in path.lower():
        instruction = "What is in this image?"
        output = "This image shows an orange plastic water jug, a common indicator of water transport and storage in Somali communities."
    elif "queue" in path.lower():
        instruction = "Analyze the water situation in this image."
        output = "The image shows a queue of people waiting for water, indicating potential water scarcity or distribution challenges in the area."
    else:
        instruction = "Describe this image."
        output = "This image provides context for water resource management in a local community."
    
    messages = [
        {"role": "user", "content": [{"type": "image", "image": path}, {"type": "text", "text": instruction}]},
        {"role": "assistant", "content": [{"type": "text", "text": output}]},
    ]
    dataset.append({"messages": messages})

if dataset:
    print(f"Dataset prepared with {len(dataset)} examples.")

## 4. Fine-Tune Gemma 4 Vision with Unsloth

In [ ]:
from unsloth import FastLanguageModel, FastVisionModel
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = True,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "current_device",
    random_state = 3407,
)

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 2,
    max_steps = 10,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    dataset_text_field = "",
    dataset_kwargs = {"skip_prepare_dataset": True},
    max_length = 2048,
    args = training_args,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

import matplotlib.pyplot as plt
logs = trainer.state.log_history
train_losses = [entry.get("loss") for entry in logs if "loss" in entry]
train_steps = [entry.get("step") for entry in logs if "loss" in entry]
if train_losses:
    plt.figure(figsize=(10, 6))
    plt.plot(train_steps, train_losses, label="Training Loss")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.title("Gemma 4 Fine-tuning Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

## 5. Inference Test & Visualizations

In [ ]:
import torch
import transformers
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
model.to("cuda")
model.eval()
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from transformers import AutoProcessor

chat_template_str = (
    "{% for message in messages %}"
    "  {% if message['role'] == 'user' %}"
    "    {{ '<start_of_turn>user\n' }}"
    "    {% for item in message['content'] %}"
    "      {% if item['type'] == 'image' %}"
    "        {{ '<image>' }}"
    "      {% elif item['type'] == 'text' %}"
    "        {{ item['text'] }}"
    "      {% endif %}"
    "    {% endfor %}"
    "    {{ '<end_of_turn>\n' }}"
    "  {% elif message['role'] == 'model' %}"
    "    {{ '<start_of_turn>model\n' }}"
    "    {% for item in message['content'] %}"
    "      {% if item['type'] == 'text' %}"
    "        {{ item['text'] }}"
    "      {% endif %}"
    "    {% endfor %}"
    "    {{ '<end_of_turn>\n' }}"
    "  {% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<start_of_turn>model\n' }}{% endif %}"
)

processor = AutoProcessor.from_pretrained("unsloth/gemma-4-E2B")
processor.tokenizer.chat_template = chat_template_str
processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

model.generation_config.max_new_tokens = 128
model.generation_config.do_sample = False
model.generation_config.num_beams = 1
model.generation_config.early_stopping = True
model.generation_config.use_cache = True

print("\n--- Inference Results ---")
for i, example in enumerate(dataset):
    messages = example["messages"]
    image_path = messages[0]["content"][0]["image"]
    instruction = messages[0]["content"][1]["text"]
    original_output = messages[1]["content"][0]["text"]
    try:
        img = Image.open(image_path).convert("RGB")
        display(img)
        inference_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": instruction}
                ]
            }
        ]
        encoded_inputs = processor(
            messages=inference_messages,
            images=img,
            return_tensors="pt",
            padding=True,
            add_generation_prompt=True
        ).to("cuda")
        
        outputs = model.generate(
            **encoded_inputs,
            max_new_tokens=128,
            use_cache=True
        )
        
        if outputs is not None:
            model_output = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
            print(f"\n--- Example {i+1} ---")
            print(f"Image: {image_path}")
            print(f"Instruction: {instruction}")
            print(f"Expected Output: {original_output}")
            print(f"Gemma 4 Output: {model_output}")
        else:
            print(f"Error: Model generation returned None for {image_path}")
    except Exception as e:
        print(f"Error processing {image_path}: {e}")